# Cleaner Colab Quickstart

이 노트북은 Cleaner MVP를 Colab에서 빠르게 실행하기 위한 환경을 만듭니다.

실행 흐름:

```text
프로젝트 ZIP 업로드 또는 GitHub clone
→ 의존성 설치
→ demo_photos 생성
→ 테스트 실행
→ Streamlit 앱 실행
→ Colab proxy URL 열기
```

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

print("Python:", sys.version)
print("Working directory:", Path.cwd())
print(subprocess.getoutput("nvidia-smi -L") or "No NVIDIA GPU detected. CPU mode is OK, but embedding will be slower.")

## 1. 프로젝트 준비

아래 셀은 두 가지 방식을 지원합니다.

1. `REPO_URL`에 GitHub 주소를 넣으면 clone합니다.
2. `REPO_URL`을 비워두면 ZIP 파일 업로드 창이 뜹니다. 이 채팅에서 받은 `Cleaner-colab-ready.zip`을 업로드하면 됩니다.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile
import subprocess

REPO_URL = ""  # 예: "https://github.com/your-name/Cleaner.git"
PROJECT_DIR = Path("/content/Cleaner")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files

    print("Upload Cleaner-colab-ready.zip or your GitHub source ZIP.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("ZIP 파일을 업로드해야 합니다.")

    upload_zip = Path(zip_names[0])
    extract_root = Path("/content/cleaner_upload")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(upload_zip) as zf:
        zf.extractall(extract_root)

    candidates = [
        p.parent
        for p in extract_root.rglob("app.py")
        if (p.parent / "src" / "cleaner").exists()
    ]
    if not candidates:
        raise RuntimeError("ZIP 안에서 Cleaner 프로젝트를 찾지 못했습니다. app.py와 src/cleaner가 있는지 확인하세요.")

    shutil.copytree(candidates[0], PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("PROJECT_DIR =", PROJECT_DIR)
print("Files:", sorted([p.name for p in PROJECT_DIR.iterdir()])[:20])

## 2. 의존성 설치

Colab에는 PyTorch가 미리 설치되어 있는 경우가 많습니다. 아래 셀은 Cleaner와 필요한 패키지를 설치합니다.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print("Installed Cleaner.")

## 3. Colab용 환경 변수와 데모 사진 준비

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_DIR = Path("/content/Cleaner")
os.chdir(PROJECT_DIR)

has_gpu = bool(subprocess.getoutput("nvidia-smi -L"))
os.environ["CLEANER_DATA_DIR"] = "/content/cleaner_data"
os.environ["CLEANER_DEFAULT_PHOTO_DIR"] = str(PROJECT_DIR / "demo_photos")
os.environ["CLEANER_DEVICE"] = "auto"
os.environ["CLEANER_BATCH_SIZE"] = "32" if has_gpu else "8"

subprocess.run([
    "python",
    "scripts/create_demo_photos.py",
    "--out",
    "demo_photos",
    "--count",
    "120",
], check=True)

print("CLEANER_DATA_DIR=", os.environ["CLEANER_DATA_DIR"])
print("CLEANER_DEFAULT_PHOTO_DIR=", os.environ["CLEANER_DEFAULT_PHOTO_DIR"])
print("CLEANER_BATCH_SIZE=", os.environ["CLEANER_BATCH_SIZE"])

## 4. 기본 테스트 실행

In [ ]:
import subprocess
subprocess.run(["pytest", "-q"], check=True)

## 5. Streamlit 앱 실행

아래 셀을 실행하면 Colab proxy URL이 출력됩니다. 출력된 URL을 열면 Cleaner 앱이 뜹니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import time

PROJECT_DIR = Path("/content/Cleaner")
os.chdir(PROJECT_DIR)

# Stop previous Streamlit process if it exists.
subprocess.run("pkill -f 'streamlit run app.py' || true", shell=True)

port = 8501
log_path = Path("/content/cleaner_streamlit.log")
log_file = log_path.open("w")
cmd = [
    "streamlit",
    "run",
    "app.py",
    "--server.address",
    "0.0.0.0",
    "--server.port",
    str(port),
    "--server.headless",
    "true",
    "--browser.gatherUsageStats",
    "false",
]
proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(5)

from google.colab import output
url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
print("Cleaner URL:", url)
print("Log file:", log_path)

## 6. 앱에서 테스트할 순서

1. 사진 폴더가 `/content/Cleaner/demo_photos`인지 확인합니다.
2. 왼쪽 사이드바에서 `임베딩 생성/업데이트`를 누릅니다.
3. `스와이프 정리` 탭에서 20장 이상을 `남김=1`, `버림=0`으로 나눕니다.
4. keep/discard가 모두 생겼다면 사이드바에서 `내 AI 학습시키기`를 누릅니다.
5. `AI 추천 검수` 탭에서 삭제 후보를 확인합니다.

첫 embedding 생성 때 pretrained model weight를 내려받기 때문에 시간이 걸릴 수 있습니다.

In [ ]:
# Optional: Streamlit log 확인
print(Path("/content/cleaner_streamlit.log").read_text()[-4000:])

In [ ]:
# Optional: 서버 중지
# proc.terminate()
# print("Stopped Streamlit server.")